<!-- NB_DOC_INTRO_V1 -->
# Tests et qualite du code ML

> 📚 **Doc thematique** : [docs/08_MLOPS.md](docs/08_MLOPS.md) (MLOps)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

Le code data science / ML doit etre teste comme tout autre code de prod. **Sans tests**, on ne sait pas quand on a casse une feature, on ne peut pas refactorer en confiance, on accumule de la dette technique a vitesse exponentielle.

Ce notebook couvre la **stack de qualite Python 2024-2025** :

1. **pytest** — framework de test (fixtures, parametrize, marks, conftest)
2. **hypothesis** — property-based testing (genere des inputs aleatoires pour tester des INVARIANTS)
3. **pandera** — validation runtime des DataFrames (typing strict + business rules)
4. **great_expectations** — validation des donnees en pipeline (assertions versionables)
5. **ruff** — lint + format ultra rapide (remplace black + isort + flake8)
6. **mypy** — type checking statique
7. **pre-commit** — hooks Git pour automatiser tout ce qui precede
8. **coverage** — mesurer la couverture des tests

Specificites ML couvertes :
- Tests d'invariance (modele insensible a la permutation des colonnes)
- Tests de data leakage (scaler ne fit pas sur le test)
- Tests de determinisme (seed → meme resultat)
- Tests de regression metrique (le nouveau modele est-il pire ?)
- Validation de DataFrames en entree de fonction (pandera)

## Plan

1. **pytest** — patterns de base, fixtures, parametrize
2. **Tests specifiques ML** — invariance, leakage, determinisme, baseline
3. **hypothesis** — property-based testing
4. **pandera** — validation runtime de DataFrames
5. **great_expectations** — validation pipeline (mention)
6. **ruff + mypy** — lint + types
7. **pre-commit** — orchestrer le tout en hooks Git
8. **coverage** — mesurer la couverture
9. **CI/CD GitHub Actions** — template pret a coller
10. Pieces et anti-patterns
11. References


## 1. pytest — patterns de base

`pytest` est le framework de facto pour tests Python (>> unittest). API simple, fixtures puissantes, discovery automatique.

### Conventions
- Fichiers : `test_*.py` ou `*_test.py`
- Fonctions : `def test_*():`
- Classes : `class Test*:` (sans `__init__`)
- Assertions : `assert <expr>` (rewrite automatique pour bon affichage des erreurs)

### Structure typique
```
project/
├── src/myproject/
│   ├── data/preprocessing.py
│   └── models/train.py
└── tests/
    ├── conftest.py         # fixtures partagees
    ├── unit/
    │   └── test_preprocessing.py
    └── integration/
        └── test_pipeline.py
```

### Exemples executables (dans ce notebook)

On va creer un mini-module et le tester directement.


In [ ]:
# pip install -q pytest hypothesis pandera ruff mypy
import pytest
import hypothesis
import sys
print(f"pytest    : {pytest.__version__}")
print(f"hypothesis: {hypothesis.__version__}")
print(f"sys.version: {sys.version}")

In [ ]:
# === Module a tester ===
import numpy as np
import pandas as pd

def normalize(arr):
    """Normalise (z-score) un array 1D ou 2D (par colonne)."""
    arr = np.asarray(arr, dtype=float)
    mean = arr.mean(axis=0)
    std = arr.std(axis=0)
    std = np.where(std == 0, 1.0, std)  # eviter div par 0
    return (arr - mean) / std

def safe_log(x, eps=1e-10):
    """Log naturel, robuste aux negatifs/zeros (clamp a eps)."""
    return np.log(np.maximum(x, eps))

def split_train_test(df, test_size=0.2, seed=42):
    """Split deterministe d'un DataFrame."""
    rng = np.random.default_rng(seed)
    n = len(df)
    n_test = int(n * test_size)
    idx = rng.permutation(n)
    return df.iloc[idx[n_test:]].reset_index(drop=True), df.iloc[idx[:n_test]].reset_index(drop=True)

# Quick check manuel
arr = np.array([1, 2, 3, 4, 5])
print(f"normalize([1..5]) = {normalize(arr)}")
print(f"safe_log([-1, 0, 1]) = {safe_log(np.array([-1, 0, 1]))}")

### 1.1 Premier test (pytest minimal)

On ne lance pas pytest CLI ici, on appelle les fonctions de test directement (equivalent du `pytest::test_X` qui passe).


In [ ]:
# === Tests basiques ===
def test_normalize_zero_mean_unit_std():
    """Apres normalize, mean=0 et std=1 sur chaque colonne."""
    arr = np.random.randn(100, 5)
    norm = normalize(arr)
    np.testing.assert_allclose(norm.mean(axis=0), 0, atol=1e-10)
    np.testing.assert_allclose(norm.std(axis=0), 1, atol=1e-10)

def test_normalize_handles_constant_column():
    """Une colonne constante (std=0) ne doit pas causer de NaN."""
    arr = np.array([[1, 5], [1, 6], [1, 7]])
    norm = normalize(arr)
    assert not np.isnan(norm).any(), "NaN dans le resultat"

def test_safe_log_handles_negatives():
    """safe_log d'un negatif ne crash pas et renvoie un float fini."""
    out = safe_log(np.array([-5, -1, 0, 1, 10]))
    assert np.all(np.isfinite(out))

def test_split_deterministic():
    """Meme seed = meme split."""
    df = pd.DataFrame({"x": range(100)})
    tr1, te1 = split_train_test(df, seed=42)
    tr2, te2 = split_train_test(df, seed=42)
    pd.testing.assert_frame_equal(tr1, tr2)
    pd.testing.assert_frame_equal(te1, te2)

# Run tests inline (simule pytest)
for name, func in list(globals().items()):
    if name.startswith("test_") and callable(func):
        try:
            func()
            print(f"  PASS  {name}")
        except AssertionError as e:
            print(f"  FAIL  {name} : {e}")

### 1.2 Fixtures pytest

Les fixtures = setup partage entre tests. Elles factorisent la creation de donnees, modeles, connexions.


In [ ]:
# conftest.py (en realite dans tests/conftest.py)

import pytest

@pytest.fixture
def sample_df():
    """Petit dataset reproductible."""
    np.random.seed(42)
    return pd.DataFrame({
        "x1": np.random.randn(100),
        "x2": np.random.randn(100),
        "y":  np.random.randint(0, 2, 100),
    })

@pytest.fixture(scope="module")
def trained_model(sample_df):
    """Modele entraine une seule fois par module test.

    Scope :
    - 'function' (default) : recree pour chaque test
    - 'class'              : 1x par classe de test
    - 'module'             : 1x par fichier
    - 'session'            : 1x pour tout pytest
    """
    from sklearn.ensemble import RandomForestClassifier
    clf = RandomForestClassifier(n_estimators=10, random_state=42)
    clf.fit(sample_df[["x1", "x2"]], sample_df["y"])
    return clf

# Demo d'usage manuel
df = sample_df.__wrapped__()                    # acceder a la fonction sous le decorateur
print(f"sample_df shape : {df.shape}")
print(df.head(3))

### 1.3 Parametrize — tester plusieurs cas


In [ ]:
# Exemple : tester normalize sur plusieurs tailles d'array
@pytest.mark.parametrize("shape", [(10,), (50, 3), (100, 1), (1000, 5)])
def test_normalize_shapes(shape):
    arr = np.random.randn(*shape)
    norm = normalize(arr)
    assert norm.shape == arr.shape
    # mean ~ 0 (avec tolerance)
    assert abs(norm.mean()) < 1e-10

# Demo manuel
for shape in [(10,), (50, 3), (100, 1), (1000, 5)]:
    arr = np.random.randn(*shape)
    norm = normalize(arr)
    assert norm.shape == arr.shape
    print(f"  PASS  shape={shape}")

### 1.4 Marks et selection de tests

Les **marks** permettent de tagger les tests et de les filtrer.

```bash
pytest -m "slow"           # ne run que les tests marques slow
pytest -m "not slow"        # exclure slow
pytest -k "normalize"       # filtre sur le nom
```

Marks built-in utiles :
- `@pytest.mark.skip(reason="...")` : skip
- `@pytest.mark.skipif(sys.platform == "win32")` : skip conditionnel
- `@pytest.mark.xfail(reason="bug #123")` : test attendu en echec
- `@pytest.mark.parametrize(...)` : multiples cas


## 2. Tests specifiques ML — invariance, leakage, determinisme

Tests ML = tests classiques + quelques patterns specifiques pour eviter les bugs ML les plus communs.


In [ ]:
# === Tests d'invariance : le modele doit etre invariant sous certaines transformations ===

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

def test_rf_invariant_under_column_permutation():
    """Un RandomForest doit donner les memes predictions si on permute l'ordre des colonnes
    (du moins en classifieur, sur le meme dataset)."""
    np.random.seed(42)
    X = np.random.randn(100, 5)
    y = (X[:, 0] + X[:, 2] > 0).astype(int)

    perm = [2, 0, 4, 1, 3]
    clf1 = RandomForestClassifier(n_estimators=10, random_state=42).fit(X, y)
    clf2 = RandomForestClassifier(n_estimators=10, random_state=42).fit(X[:, perm], y)

    # Predictions sur le meme X permute
    p1 = clf1.predict(X)
    p2 = clf2.predict(X[:, perm])
    # Note : avec random_state fixe, RF a un determinisme sur la structure interne,
    # mais l'ordre des colonnes peut affecter le tie-breaking. On accepte une difference faible.
    diff_rate = (p1 != p2).mean()
    assert diff_rate < 0.05, f"Trop de difference apres permutation : {diff_rate:.3f}"
    print(f"  PASS  invariant_under_permutation : diff_rate={diff_rate:.3f}")

test_rf_invariant_under_column_permutation()

### 2.1 Test anti data-leakage

C'est LE bug le plus grave en ML : le scaler est fit sur tout (train + test), puis on split. Resultat : le test "voit" les statistiques du futur → score surevaluee.


In [ ]:
def test_pipeline_no_data_leakage():
    """Le scaler dans une Pipeline ne doit pas se refit quand on appelle predict.

    On verifie que la mean du scaler ne change pas apres avoir appele predict sur test."""
    np.random.seed(42)
    X_train = np.random.randn(80, 3)
    X_test = np.random.randn(20, 3) * 100 + 1000   # test tres different
    y_train = np.random.randint(0, 2, 80)

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    LogisticRegression(max_iter=100)),
    ])
    pipe.fit(X_train, y_train)
    mean_after_fit = pipe.named_steps["scaler"].mean_.copy()

    # predict ne doit PAS refit le scaler
    pipe.predict(X_test)
    mean_after_predict = pipe.named_steps["scaler"].mean_.copy()

    np.testing.assert_array_equal(mean_after_fit, mean_after_predict)
    print("  PASS  no_data_leakage : scaler.mean_ inchange apres predict")

test_pipeline_no_data_leakage()

### 2.2 Test de determinisme avec seed


In [ ]:
def test_rf_deterministic():
    """Avec random_state=42, deux runs identiques donnent les memes predictions."""
    np.random.seed(0)  # generation des donnees aussi seeded
    X = np.random.randn(50, 4)
    y = np.random.randint(0, 2, 50)

    pred1 = RandomForestClassifier(random_state=42).fit(X, y).predict(X)
    pred2 = RandomForestClassifier(random_state=42).fit(X, y).predict(X)

    np.testing.assert_array_equal(pred1, pred2)
    print("  PASS  rf_deterministic")

test_rf_deterministic()

### 2.3 Test "modele bat la baseline"

A toujours faire : sans ce test, on peut deployer un modele pire qu'un dummy.


In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.dummy import DummyClassifier

def test_model_beats_majority_baseline():
    """Le modele doit faire mieux qu'un dummy majoritaire de >= 5 points."""
    np.random.seed(42)
    # Donnees avec un signal (X[:, 0] correle a y)
    X = np.random.randn(500, 5)
    y = (X[:, 0] + 0.3 * np.random.randn(500) > 0).astype(int)

    clf = RandomForestClassifier(n_estimators=50, random_state=42).fit(X[:400], y[:400])
    acc = accuracy_score(y[400:], clf.predict(X[400:]))

    dummy = DummyClassifier(strategy="prior").fit(X[:400], y[:400])
    acc_dummy = accuracy_score(y[400:], dummy.predict(X[400:]))

    assert acc > acc_dummy + 0.05, f"Modele ({acc:.3f}) <= baseline ({acc_dummy:.3f}) + 0.05"
    print(f"  PASS  beats_baseline : model={acc:.3f}, dummy={acc_dummy:.3f}")

test_model_beats_majority_baseline()

### 2.4 Test de regression metrique (golden tests)

Eviter de regresser : lors du commit, le score doit etre >= au dernier baseline connu.


In [ ]:
GOLDEN_SCORES = {
    "accuracy_iris_rf_42": 0.95,   # le score qu'on attend / dernier baseline
}

def test_iris_rf_does_not_regress():
    """Iris + RF + seed=42 doit donner >= 0.95 d'accuracy."""
    from sklearn.datasets import load_iris
    from sklearn.model_selection import cross_val_score
    X, y = load_iris(return_X_y=True)
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    score = cross_val_score(clf, X, y, cv=5).mean()
    expected = GOLDEN_SCORES["accuracy_iris_rf_42"]
    assert score >= expected, f"Regression : got {score:.3f} < {expected}"
    print(f"  PASS  iris_no_regression : {score:.3f} >= {expected}")

test_iris_rf_does_not_regress()

## 3. hypothesis — property-based testing

Plutot que de tester des **cas d'usage individuels**, hypothesis genere automatiquement **des centaines** d'inputs aleatoires et verifie qu'une **propriete** tient pour tous. Decouvre des edge cases qu'on n'aurait jamais imagines.

### Exemple : tester la propriete "double normalize = normalize"


In [ ]:
from hypothesis import given, strategies as st, settings, HealthCheck

@given(
    arr=st.lists(
        st.floats(min_value=-100, max_value=100, allow_nan=False, allow_infinity=False),
        min_size=10, max_size=200,
    )
)
@settings(max_examples=30, suppress_health_check=[HealthCheck.too_slow])
def test_normalize_idempotent(arr):
    """Property : normalize(normalize(x)) ~~ normalize(x) (idempotent)."""
    a = np.array(arr)
    norm1 = normalize(a)
    norm2 = normalize(norm1)
    # Tolerance car std de norm1 est deja 1 -> normalize(norm1) = norm1 (sauf precision flottante)
    np.testing.assert_allclose(norm1, norm2, atol=1e-10)

# Run
test_normalize_idempotent()
print("  PASS  normalize_idempotent (30 random inputs tested)")

### 3.1 Propriete d'invariance sous translation


In [ ]:
@given(
    arr=st.lists(st.floats(min_value=-100, max_value=100, allow_nan=False), min_size=10, max_size=100),
    shift=st.floats(min_value=-50, max_value=50, allow_nan=False),
)
@settings(max_examples=20)
def test_normalize_invariant_under_translation(arr, shift):
    """normalize(x + c) == normalize(x) (l'ajout d'une constante ne change pas le z-score)."""
    a = np.array(arr)
    norm_a = normalize(a)
    norm_a_shifted = normalize(a + shift)
    np.testing.assert_allclose(norm_a, norm_a_shifted, atol=1e-8)

test_normalize_invariant_under_translation()
print("  PASS  invariant_under_translation (20 random pairs tested)")

### 3.2 Hypothesis decouvre des edge cases

Si une propriete echoue, hypothesis affiche le **plus petit** input qui casse (shrinking).


In [ ]:
# Test PROVOQUE pour montrer le shrinking
# Ici on a une fonction buggee : divise par x sans gerer 0
def divide_by(x):
    return 1 / x

@given(x=st.floats(allow_nan=False, allow_infinity=False))
@settings(max_examples=20, suppress_health_check=[HealthCheck.filter_too_much])
def test_divide_buggy(x):
    # On va tomber sur x=0 -> ZeroDivisionError
    if abs(x) < 0.001:
        return  # skip ces cas, sinon le test echoue (mais c'est la le bug !)
    result = divide_by(x)
    assert isinstance(result, float)

# Avec le filter, le test passe. Si on l'enleve, hypothesis trouverait x=0.0 :
print("  Demo de filtrage : hypothesis aurait trouve x=0.0 sans le filtre")

test_divide_buggy()
print("  PASS  divide_buggy (avec filtre evidemment)")

## 4. pandera — validation runtime de DataFrames

Pandera permet de declarer un **schema** pour un DataFrame (types, contraintes, business rules) et de le **valider au runtime**.

> 🎯 **Use case typique** : fonction qui prend un DataFrame en entree. Au lieu de faire des `assert` partout, on declare le schema attendu.


In [ ]:
# pip install -q pandera
import pandera as pa
from pandera import Column, DataFrameSchema, Check

# === Definition du schema ===
transaction_schema = DataFrameSchema({
    "id":       Column(int, Check.ge(0), unique=True),
    "amount":   Column(float, [Check.ge(0), Check.le(1_000_000)]),
    "country":  Column(str, Check.isin(["FR", "DE", "ES", "IT"])),
    "date":     Column(pa.DateTime),
    "category": Column(str, nullable=True),
})

# === DataFrame valide ===
df_valid = pd.DataFrame({
    "id":       [1, 2, 3],
    "amount":   [100.0, 250.5, 999.99],
    "country":  ["FR", "DE", "FR"],
    "date":     pd.to_datetime(["2024-01-01", "2024-01-02", "2024-01-03"]),
    "category": ["food", None, "transport"],
})

try:
    transaction_schema.validate(df_valid)
    print("  PASS  validation df_valid")
except pa.errors.SchemaError as e:
    print(f"  FAIL  : {e}")

In [ ]:
# === DataFrame INVALIDE (amount negatif) ===
df_bad = df_valid.copy()
df_bad.loc[0, "amount"] = -50.0       # negatif !

try:
    transaction_schema.validate(df_bad, lazy=True)   # lazy=True : reporte TOUTES les erreurs
    print("  FAIL  : aurait du detecter le bug")
except pa.errors.SchemaErrors as e:
    print(f"  PASS  : erreur detectee comme attendu")
    print(f"  Nombre de violations : {len(e.failure_cases)}")
    print(e.failure_cases.head())

### 4.1 Decorateur pour valider input/output de fonctions


In [ ]:
@pa.check_input(transaction_schema)
def process_transactions(df: pd.DataFrame) -> pd.DataFrame:
    """Cette fonction VALIDE son input automatiquement au call."""
    return df.assign(amount_ttc=df["amount"] * 1.2)

# OK
result = process_transactions(df_valid)
print(f"  PASS  process_transactions(df_valid) : shape={result.shape}")

# KO : devrait raise
try:
    process_transactions(df_bad)
    print("  FAIL  : aurait du crash sur df_bad")
except pa.errors.SchemaError:
    print(f"  PASS  : a crash comme attendu sur df_bad")

### 4.2 Schemas avec model class (style Pydantic)

API plus moderne, comme Pydantic mais pour DataFrames.


In [ ]:
from pandera.typing import Series, DataFrame
import pandera.api.pandas.model as pmodel

class TransactionModel(pmodel.DataFrameModel):
    id:       Series[int]   = pa.Field(ge=0, unique=True)
    amount:   Series[float] = pa.Field(ge=0, le=1_000_000)
    country:  Series[str]   = pa.Field(isin=["FR", "DE", "ES", "IT"])

    class Config:
        strict = True   # erreur si colonne en plus

# Validation
try:
    TransactionModel.validate(df_valid.drop(columns=["date", "category"]))
    print("  PASS  TransactionModel.validate")
except pa.errors.SchemaError as e:
    print(f"  FAIL  : {e}")

## 5. great_expectations — validation pipeline (mention)

Quand on a un pipeline de donnees (Airflow / Prefect / Dagster), `great_expectations` permet de definir des **suites d'expectations** versionnees (YAML) appliquees a chaque batch.

Difference avec pandera :
- **pandera** : validation programmatique, dans le code Python
- **great_expectations** : suite versionnable, generation de **data docs** (HTML), integration native Airflow

```bash
pip install -q great-expectations
great_expectations init     # initialise le projet
```

Exemple d'expectation :
```python
# import great_expectations as gx
# context = gx.get_context()
# validator = context.sources.pandas_default.read_csv("data.csv")
# validator.expect_column_to_exist("amount")
# validator.expect_column_values_to_be_between("amount", min_value=0, max_value=1_000_000)
# validator.expect_column_values_to_not_be_null("user_id")
# validator.save_expectation_suite()
```

Pour ce notebook on utilise pandera, plus leger.


## 6. Ruff + mypy — lint + types

### Ruff
**Ruff** (Astral) est le linter / formatter Python le plus rapide (Rust). Remplace `black + isort + flake8 + pyupgrade + ...` en une seule commande.

Installation :
```bash
uv add --group dev ruff
# ou : pip install -q ruff
```

Usage :
```bash
ruff check .                 # lint
ruff check . --fix           # lint + corrige automatiquement les warnings simples
ruff format .                # format (style black)
```

Configuration dans `pyproject.toml` :
```toml
[tool.ruff]
line-length = 100
target-version = "py311"

[tool.ruff.lint]
select = ["E", "F", "I", "B", "UP", "N", "W"]
ignore = ["E501"]

[tool.ruff.format]
quote-style = "double"
```

### mypy — type checking


In [ ]:
# Exemple : code avec annotations qui plairait a mypy
from typing import Protocol, Sequence
import numpy.typing as npt

class Estimator(Protocol):
    """Protocol pour duck-typing structurel."""
    def fit(self, X: npt.NDArray, y: npt.NDArray) -> "Estimator": ...
    def predict(self, X: npt.NDArray) -> npt.NDArray: ...

def train_and_score(
    model: Estimator,
    X_train: npt.NDArray[np.float64],
    y_train: npt.NDArray[np.int_],
    X_test: npt.NDArray[np.float64],
    y_test: npt.NDArray[np.int_],
) -> float:
    """Type-annoted function. mypy verifierait que model.fit retourne bien Estimator,
    que X_train est bien un array float64, etc."""
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    return float((pred == y_test).mean())

# Test
np.random.seed(42)
Xtr = np.random.randn(80, 3)
ytr = np.random.randint(0, 2, 80)
Xte = np.random.randn(20, 3)
yte = np.random.randint(0, 2, 20)

acc = train_and_score(RandomForestClassifier(n_estimators=10, random_state=42), Xtr, ytr, Xte, yte)
print(f"  PASS  type-annoted function works : acc={acc:.3f}")

## 7. pre-commit — hooks Git automatises

[pre-commit](https://pre-commit.com/) execute des hooks **avant chaque commit**. Empeche de pusher du code mal formate / non type / avec outputs notebooks.

### Installation
```bash
uv add --group dev pre-commit
uv run pre-commit install      # installe les hooks dans .git/hooks/
```

### `.pre-commit-config.yaml`
```yaml
repos:
  - repo: https://github.com/astral-sh/ruff-pre-commit
    rev: v0.6.0
    hooks:
      - id: ruff
        args: [--fix]
      - id: ruff-format

  - repo: https://github.com/pre-commit/pre-commit-hooks
    rev: v4.6.0
    hooks:
      - id: trailing-whitespace
      - id: end-of-file-fixer
      - id: check-yaml
      - id: check-added-large-files
      - id: check-merge-conflict

  - repo: https://github.com/pre-commit/mirrors-mypy
    rev: v1.11.0
    hooks:
      - id: mypy
        additional_dependencies: [types-requests, pandas-stubs]

  - repo: https://github.com/kynan/nbstripout
    rev: 0.7.1
    hooks:
      - id: nbstripout      # nettoie les outputs des notebooks avant commit
```

### Usage
```bash
git commit ...                          # hooks tournent automatiquement
pre-commit run --all-files               # run sur TOUT le repo
pre-commit autoupdate                    # MAJ les versions des hooks
```


## 8. Coverage — mesurer la couverture des tests


In [ ]:
# pip install -q pytest-cov coverage

# === Usage ===
# pytest --cov=src --cov-report=html --cov-report=term-missing

# Configuration dans pyproject.toml :
toml_excerpt = '''
[tool.coverage.run]
branch = true
source = ["src"]
omit = ["tests/*", "*/test_*"]

[tool.coverage.report]
fail_under = 80          # echoue le CI si < 80% couverture
show_missing = true
exclude_lines = [
    "pragma: no cover",
    "def __repr__",
    "raise NotImplementedError",
    "if __name__ == .__main__.:",
]
'''
print(toml_excerpt)

### Couverture par branches

`branch=true` mesure aussi les branchements (`if`, `for`, `while`) : un code avec 100% de lines mais 0% de branches a une couverture trompeuse.

> 🎯 **Cible realiste** : 80% de couverture sur code de prod. Pas besoin de 100% (effort decroissant).


## 9. CI/CD GitHub Actions — template pret a coller

`.github/workflows/ci.yml` :

```yaml
name: CI
on:
  push:
    branches: [main]
  pull_request:

jobs:
  test:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: ["3.11", "3.12"]

    steps:
      - uses: actions/checkout@v4

      - name: Install uv
        uses: astral-sh/setup-uv@v3

      - name: Set up Python
        run: uv python install ${{ matrix.python-version }}

      - name: Install dependencies
        run: uv sync --group dev --group ml

      - name: Lint with ruff
        run: uv run ruff check .

      - name: Format check
        run: uv run ruff format --check .

      - name: Type check with mypy
        run: uv run mypy src/

      - name: Tests with coverage
        run: uv run pytest --cov=src --cov-report=xml --cov-fail-under=80

      - name: Upload coverage
        uses: codecov/codecov-action@v4
        with:
          file: ./coverage.xml

      - name: Test notebooks
        run: uv run pytest --nbmake notebooks/*.ipynb
```


## 10. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correction |
|---|---|
| Tester juste les modeles, pas les utils | Tester chaque fonction critique |
| Pas de seed → tests flaky | `random_state=42` partout |
| Test sur le vrai dataset (lent, ne marche pas en CI sans donnees) | Fixture avec sample synth |
| Mock le modele pour tester le wiring | Utiliser un vrai modele leger (RF n_estimators=5) |
| 100% coverage mais 0 assertion | Verifier que les tests ont des `assert` |
| Tester l'implementation (`self._n_estimators`) | Tester le comportement (`predict()` shape) |
| `if __name__ == "__main__": ...` pour run | Vraie suite pytest avec discovery |
| Pas de tests d'integration end-to-end | Au moins 1 test "fit → predict sur tiny data" |
| Coverage requise 100% partout | 80% sur code de prod, 60% sur scripts utilitaires |
| Pre-commit hooks trop lents | Les mettre en pre-push, lints rapides en pre-commit |
| Pas tester les notebooks | `pytest --nbmake notebooks/*.ipynb` |
| Catch tout exception sans assertion | `pytest.raises(ValueError, match="invalid")` |

### Test "table" — un seul test parametrize au lieu de 10 tests presque identiques


In [ ]:
# Pattern recommande : table-driven tests
test_cases = [
    # (input, expected_output, description)
    ([1, 2, 3, 4, 5],           "z-score classique",   None),
    ([0, 0, 0, 0, 0],           "tous zeros",          None),
    ([1.5, 2.5, -3.0, 100.0],   "avec outlier",        None),
    ([-1e9, 0, 1e9],            "extreme range",       None),
]

for arr, desc, _ in test_cases:
    norm = normalize(arr)
    assert not np.isnan(norm).any(), f"NaN dans {desc}"
    print(f"  PASS  {desc}")

## 11. References

### Documentation officielle
- **pytest** : https://docs.pytest.org/
- **hypothesis** : https://hypothesis.readthedocs.io/
- **pandera** : https://pandera.readthedocs.io/
- **great_expectations** : https://docs.greatexpectations.io/
- **ruff** : https://docs.astral.sh/ruff/
- **mypy** : https://mypy.readthedocs.io/
- **pre-commit** : https://pre-commit.com/
- **coverage** : https://coverage.readthedocs.io/

### Livres et papers
- *Effective Python Testing With Pytest* — Brian Okken (livre)
- *Property-Based Testing with PropEr, Erlang, and Elixir* — Fred Hebert (concepts)
- *Beyond Coverage* — Marick (1998) — sur les limites de la couverture

### Voir aussi (collection)
- [MLOps_Tracking_DVC_Wandb.ipynb](./MLOps_Tracking_DVC_Wandb.ipynb) — versioning tests + experiences
- [MLOps_Drift_Monitoring.ipynb](./MLOps_Drift_Monitoring.ipynb) — tests stat de drift
- [DS_Metrics_Reference.ipynb](./DS_Metrics_Reference.ipynb) — metriques utilisees dans les tests de regression
